# Hull Tactical Market Prediction - AutoML Approach

Este notebook implementa una solución completa usando AutoML para la competencia Hull Tactical Market Prediction.

## Objetivo
Predecir los retornos diarios del S&P 500 usando un conjunto de características de mercado, optimizando para el Adjusted Sharpe Ratio.

## Estructura del Notebook
1. **Setup y Configuración**
2. **Análisis Exploratorio de Datos (EDA)**
3. **Implementación de Métrica de Evaluación**
4. **Ingeniería de Características**
5. **Modelos AutoML**
6. **Ensemble y Optimización**
7. **Validación y Testing**
8. **Submission**

## 1. Setup y Configuración

In [ ]:
# Instalar librerías necesarias
import subprocess
import sys

def install_package(package):
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"✅ {package} instalado exitosamente")
    except subprocess.CalledProcessError:
        print(f"❌ Error instalando {package}")

# Lista de paquetes para AutoML
packages = [
    "autogluon",
    "flaml",
    "optuna",
    "lightgbm",
    "xgboost",
    "catboost",
    "scikit-learn",
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
    "plotly",
    "ta",  # Technical Analysis library
    "yfinance"
]

print("🚀 Instalando librerías de AutoML...")
for package in packages:
    install_package(package)

In [ ]:
# Importar librerías principales
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("📚 Librerías importadas exitosamente")

In [ ]:
# Importar librerías de ML y AutoML
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# AutoML frameworks
try:
    from autogluon.tabular import TabularPredictor
    print("✅ AutoGluon importado")
except ImportError:
    print("❌ AutoGluon no disponible")

try:
    from flaml import AutoML
    print("✅ FLAML importado")
except ImportError:
    print("❌ FLAML no disponible")

import optuna
import lightgbm as lgb
import xgboost as xgb
import catboost as cb

print("🤖 Librerías de AutoML configuradas")

## 2. Carga y Análisis Exploratorio de Datos

In [ ]:
# Cargar datos de entrenamiento
try:
    train_df = pd.read_csv('/kaggle/input/hull-tactical-market-prediction/train.csv')
    print("✅ Datos cargados desde Kaggle")
except FileNotFoundError:
    # Para desarrollo local, crear datos sintéticos
    print("⚠️ Datos de Kaggle no encontrados, creando datos sintéticos para desarrollo...")
    np.random.seed(42)
    n_samples = 5000
    n_features = 50
    
    # Crear características sintéticas que simulan datos financieros
    data = {}
    data['date_id'] = range(n_samples)
    
    # Features técnicos simulados
    for i in range(n_features):
        if i < 10:  # Price-based features
            data[f'feature_{i}'] = np.random.normal(0, 1, n_samples) + np.sin(np.arange(n_samples) * 0.01)
        elif i < 20:  # Volume-based features
            data[f'feature_{i}'] = np.random.exponential(1, n_samples)
        elif i < 30:  # Volatility features
            data[f'feature_{i}'] = np.random.gamma(2, 2, n_samples)
        else:  # Other market indicators
            data[f'feature_{i}'] = np.random.normal(0, 2, n_samples)
    
    # Target variable (forward returns)
    # Simulamos retornos con algo de predictibilidad
    signal = (data['feature_0'] * 0.1 + data['feature_5'] * 0.05 + 
              data['feature_10'] * -0.03 + np.random.normal(0, 0.02, n_samples))
    data['forward_return_1d'] = signal
    
    train_df = pd.DataFrame(data)
    print(f"📊 Datos sintéticos creados: {train_df.shape}")

print(f"📈 Forma de los datos: {train_df.shape}")
print(f"📅 Rango de fechas: {train_df['date_id'].min()} - {train_df['date_id'].max()}")
train_df.head()

In [ ]:
# Información básica del dataset
print("📊 INFORMACIÓN BÁSICA DEL DATASET")
print("=" * 50)
print(f"Número de filas: {train_df.shape[0]:,}")
print(f"Número de columnas: {train_df.shape[1]:,}")
print(f"Memoria utilizada: {train_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Identificar columnas de características
feature_cols = [col for col in train_df.columns if col.startswith('feature_')]
target_col = 'forward_return_1d'

print(f"\n🎯 Variable objetivo: {target_col}")
print(f"🔢 Número de características: {len(feature_cols)}")

# Estadísticas de la variable objetivo
print(f"\n📈 ESTADÍSTICAS DE LA VARIABLE OBJETIVO")
print("=" * 50)
target_stats = train_df[target_col].describe()
print(target_stats)

# Verificar valores faltantes
missing_data = train_df.isnull().sum()
if missing_data.sum() > 0:
    print(f"\n⚠️ VALORES FALTANTES")
    print("=" * 50)
    print(missing_data[missing_data > 0])
else:
    print(f"\n✅ No hay valores faltantes en el dataset")

In [ ]:
# Visualización de la variable objetivo
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Análisis de la Variable Objetivo: Forward Return 1D', fontsize=16, fontweight='bold')

# Serie temporal
axes[0, 0].plot(train_df['date_id'], train_df[target_col], alpha=0.7, linewidth=0.8)
axes[0, 0].set_title('Serie Temporal de Retornos')
axes[0, 0].set_xlabel('Date ID')
axes[0, 0].set_ylabel('Forward Return')
axes[0, 0].grid(True, alpha=0.3)

# Histograma
axes[0, 1].hist(train_df[target_col], bins=50, alpha=0.7, edgecolor='black')
axes[0, 1].set_title('Distribución de Retornos')
axes[0, 1].set_xlabel('Forward Return')
axes[0, 1].set_ylabel('Frecuencia')
axes[0, 1].grid(True, alpha=0.3)

# Q-Q plot
from scipy import stats
stats.probplot(train_df[target_col], dist="norm", plot=axes[1, 0])
axes[1, 0].set_title('Q-Q Plot (Normalidad)')
axes[1, 0].grid(True, alpha=0.3)

# Rolling statistics
rolling_mean = train_df[target_col].rolling(window=100).mean()
rolling_std = train_df[target_col].rolling(window=100).std()

axes[1, 1].plot(train_df['date_id'], rolling_mean, label='Media Móvil (100)', alpha=0.8)
axes[1, 1].plot(train_df['date_id'], rolling_std, label='Desv. Estándar Móvil (100)', alpha=0.8)
axes[1, 1].set_title('Estadísticas Móviles')
axes[1, 1].set_xlabel('Date ID')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Estadísticas adicionales
print(f"📊 ESTADÍSTICAS ADICIONALES")
print("=" * 50)
print(f"Skewness: {train_df[target_col].skew():.4f}")
print(f"Kurtosis: {train_df[target_col].kurtosis():.4f}")
print(f"Sharpe Ratio (aproximado): {train_df[target_col].mean() / train_df[target_col].std():.4f}")

## 3. Implementación de la Métrica de Evaluación (Adjusted Sharpe Ratio)

In [ ]:
def adjusted_sharpe_ratio(y_true, y_pred, max_weight=6.0):
    """
    Calcula el Adjusted Sharpe Ratio usado en la competencia Hull Tactical.
    
    La métrica penaliza estrategias que toman demasiado riesgo y premia
    aquellas que generan retornos consistentes con volatilidad controlada.
    
    Args:
        y_true: Retornos reales
        y_pred: Predicciones (posiciones/pesos)
        max_weight: Peso máximo permitido
    
    Returns:
        float: Adjusted Sharpe Ratio
    """
    # Convertir a numpy arrays
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    # Aplicar límites a las predicciones
    y_pred = np.clip(y_pred, -max_weight, max_weight)
    
    # Calcular retornos de la estrategia
    strategy_returns = y_true * y_pred
    
    # Calcular métricas
    mean_return = np.mean(strategy_returns)
    std_return = np.std(strategy_returns)
    
    # Evitar división por cero
    if std_return == 0:
        return 0.0
    
    # Sharpe ratio ajustado
    sharpe = mean_return / std_return
    
    # Factor de ajuste por volatilidad (penaliza alta volatilidad)
    volatility_penalty = 1.0 / (1.0 + std_return)
    
    adjusted_sharpe = sharpe * volatility_penalty
    
    return adjusted_sharpe

def competition_score(y_true, y_pred):
    """
    Función wrapper para usar con sklearn y AutoML frameworks
    """
    return adjusted_sharpe_ratio(y_true, y_pred)

# Función para evaluación con cross-validation temporal
def evaluate_model_temporal(model, X, y, n_splits=5):
    """
    Evalúa un modelo usando validación cruzada temporal
    """
    tscv = TimeSeriesSplit(n_splits=n_splits)
    scores = []
    
    for train_idx, val_idx in tscv.split(X):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        
        score = competition_score(y_val, y_pred)
        scores.append(score)
    
    return np.array(scores)

print("✅ Función de evaluación implementada")

# Test de la función con datos sintéticos
test_true = np.random.normal(0, 0.02, 1000)
test_pred = np.random.normal(0, 1, 1000)
test_score = adjusted_sharpe_ratio(test_true, test_pred)
print(f"🧪 Test score: {test_score:.6f}")

## 4. Ingeniería de Características

In [ ]:
def create_technical_features(df, feature_cols, target_col):
    """
    Crea características técnicas avanzadas para trading
    """
    df_enhanced = df.copy()
    
    print("🔧 Creando características técnicas...")
    
    # 1. Características de lag
    print("   📈 Lags y diferencias...")
    for col in feature_cols[:10]:  # Solo las primeras 10 para evitar sobrecarga
        for lag in [1, 2, 3, 5, 10]:
            df_enhanced[f'{col}_lag_{lag}'] = df_enhanced[col].shift(lag)
        
        # Diferencias
        df_enhanced[f'{col}_diff_1'] = df_enhanced[col].diff(1)
        df_enhanced[f'{col}_diff_5'] = df_enhanced[col].diff(5)
    
    # 2. Rolling statistics
    print("   📊 Estadísticas móviles...")
    windows = [5, 10, 20, 50]
    for col in feature_cols[:5]:  # Reducir para evitar explosión de features
        for window in windows:
            df_enhanced[f'{col}_rolling_mean_{window}'] = df_enhanced[col].rolling(window).mean()
            df_enhanced[f'{col}_rolling_std_{window}'] = df_enhanced[col].rolling(window).std()
            df_enhanced[f'{col}_rolling_min_{window}'] = df_enhanced[col].rolling(window).min()
            df_enhanced[f'{col}_rolling_max_{window}'] = df_enhanced[col].rolling(window).max()
    
    # 3. Ratios y transformaciones
    print("   🔄 Ratios y transformaciones...")
    for i, col1 in enumerate(feature_cols[:5]):
        for col2 in feature_cols[i+1:6]:  # Evitar demasiadas combinaciones
            # Ratio
            df_enhanced[f'{col1}_{col2}_ratio'] = df_enhanced[col1] / (df_enhanced[col2] + 1e-8)
            # Diferencia
            df_enhanced[f'{col1}_{col2}_diff'] = df_enhanced[col1] - df_enhanced[col2]
    
    # 4. Características de momentum
    print("   🚀 Momentum y tendencias...")
    for col in feature_cols[:5]:
        # ROC (Rate of Change)
        for period in [5, 10, 20]:
            df_enhanced[f'{col}_roc_{period}'] = (df_enhanced[col] / df_enhanced[col].shift(period) - 1) * 100
        
        # RSI aproximado
        delta = df_enhanced[col].diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
        rs = gain / loss
        df_enhanced[f'{col}_rsi'] = 100 - (100 / (1 + rs))
    
    # 5. Características de volatilidad
    print("   📉 Volatilidad...")
    for col in feature_cols[:3]:
        # Volatilidad realizada
        returns = df_enhanced[col].pct_change()
        for window in [10, 20, 50]:
            df_enhanced[f'{col}_volatility_{window}'] = returns.rolling(window).std() * np.sqrt(252)
    
    # 6. Características de target lag (cuidado con data leakage)
    print("   🎯 Target lags (históricos)...")
    for lag in [2, 3, 5, 10]:  # Empezar desde lag 2 para evitar leakage
        df_enhanced[f'target_lag_{lag}'] = df_enhanced[target_col].shift(lag)
    
    # 7. Características de tiempo
    print("   ⏰ Características temporales...")
    df_enhanced['day_of_period'] = df_enhanced['date_id'] % 252  # Aproximadamente días de trading por año
    df_enhanced['week_of_period'] = df_enhanced['date_id'] % 52
    df_enhanced['month_of_period'] = df_enhanced['date_id'] % 12
    
    # Características cíclicas
    df_enhanced['day_sin'] = np.sin(2 * np.pi * df_enhanced['day_of_period'] / 252)
    df_enhanced['day_cos'] = np.cos(2 * np.pi * df_enhanced['day_of_period'] / 252)
    
    print(f"✅ Características creadas. Forma final: {df_enhanced.shape}")
    return df_enhanced

# Aplicar ingeniería de características
print("🚀 Iniciando ingeniería de características...")
train_enhanced = create_technical_features(train_df, feature_cols, target_col)

# Eliminar filas con NaN (debido a lags y rolling)
initial_shape = train_enhanced.shape
train_enhanced = train_enhanced.dropna()
final_shape = train_enhanced.shape

print(f"📊 Forma inicial: {initial_shape}")
print(f"📊 Forma final (sin NaN): {final_shape}")
print(f"📊 Filas eliminadas: {initial_shape[0] - final_shape[0]}")

In [ ]:
# Selección de características más importantes
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression
from sklearn.ensemble import RandomForestRegressor

def select_best_features(X, y, method='random_forest', k=100):
    """
    Selecciona las mejores características usando diferentes métodos
    """
    print(f"🔍 Seleccionando las mejores {k} características usando {method}...")
    
    if method == 'random_forest':
        # Random Forest feature importance
        rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
        rf.fit(X, y)
        
        feature_importance = pd.DataFrame({
            'feature': X.columns,
            'importance': rf.feature_importances_
        }).sort_values('importance', ascending=False)
        
        selected_features = feature_importance.head(k)['feature'].tolist()
        
    elif method == 'f_regression':
        # F-test
        selector = SelectKBest(score_func=f_regression, k=k)
        selector.fit(X, y)
        selected_features = X.columns[selector.get_support()].tolist()
        
    elif method == 'mutual_info':
        # Mutual Information
        selector = SelectKBest(score_func=mutual_info_regression, k=k)
        selector.fit(X, y)
        selected_features = X.columns[selector.get_support()].tolist()
    
    return selected_features

# Preparar datos para selección de características
feature_columns = [col for col in train_enhanced.columns 
                  if col not in ['date_id', target_col]]

X = train_enhanced[feature_columns]
y = train_enhanced[target_col]

print(f"📊 Características disponibles: {len(feature_columns)}")

# Seleccionar mejores características con diferentes métodos
selected_features_rf = select_best_features(X, y, 'random_forest', k=50)
selected_features_f = select_best_features(X, y, 'f_regression', k=50)

# Combinar características seleccionadas
all_selected = list(set(selected_features_rf + selected_features_f))
print(f"🎯 Características finales seleccionadas: {len(all_selected)}")

# Crear dataset final
X_final = train_enhanced[all_selected]
y_final = train_enhanced[target_col]

print(f"✅ Dataset final preparado: {X_final.shape}")

## 5. Modelos AutoML

In [ ]:
# Preparar datos para entrenamiento
from sklearn.model_selection import train_test_split

# Split temporal (importante para series de tiempo)
split_point = int(len(X_final) * 0.8)
X_train = X_final.iloc[:split_point]
X_test = X_final.iloc[split_point:]
y_train = y_final.iloc[:split_point]
y_test = y_final.iloc[split_point:]

print(f"📊 Datos de entrenamiento: {X_train.shape}")
print(f"📊 Datos de prueba: {X_test.shape}")

# Escalado de características
scaler = RobustScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), 
    columns=X_train.columns, 
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test), 
    columns=X_test.columns, 
    index=X_test.index
)

print("✅ Datos escalados y preparados")

In [ ]:
# Diccionario para almacenar modelos y resultados
models_results = {}

def evaluate_and_store_model(name, model, X_train, y_train, X_test, y_test):
    """
    Entrena, evalúa y almacena un modelo
    """
    print(f"🤖 Entrenando {name}...")
    
    # Entrenar modelo
    model.fit(X_train, y_train)
    
    # Predicciones
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    
    # Evaluación
    train_score = competition_score(y_train, y_pred_train)
    test_score = competition_score(y_test, y_pred_test)
    
    # Métricas adicionales
    train_mse = mean_squared_error(y_train, y_pred_train)
    test_mse = mean_squared_error(y_test, y_pred_test)
    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)
    
    results = {
        'model': model,
        'train_score': train_score,
        'test_score': test_score,
        'train_mse': train_mse,
        'test_mse': test_mse,
        'train_r2': train_r2,
        'test_r2': test_r2,
        'predictions_train': y_pred_train,
        'predictions_test': y_pred_test
    }
    
    models_results[name] = results
    
    print(f"   ✅ {name} - Train Score: {train_score:.6f}, Test Score: {test_score:.6f}")
    return results

In [ ]:
# 1. Modelos baseline
print("🚀 ENTRENANDO MODELOS BASELINE")
print("=" * 50)

# Random Forest
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
evaluate_and_store_model('RandomForest', rf_model, X_train_scaled, y_train, X_test_scaled, y_test)

# Gradient Boosting
gb_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)
evaluate_and_store_model('GradientBoosting', gb_model, X_train_scaled, y_train, X_test_scaled, y_test)

# Ridge Regression
ridge_model = Ridge(alpha=1.0, random_state=42)
evaluate_and_store_model('Ridge', ridge_model, X_train_scaled, y_train, X_test_scaled, y_test)

In [ ]:
# 2. LightGBM con optimización
print("\n🚀 ENTRENANDO LIGHTGBM OPTIMIZADO")
print("=" * 50)

# LightGBM básico
lgb_model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    num_leaves=31,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
evaluate_and_store_model('LightGBM', lgb_model, X_train, y_train, X_test, y_test)

# XGBoost
xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)
evaluate_and_store_model('XGBoost', xgb_model, X_train, y_train, X_test, y_test)

# CatBoost
cat_model = cb.CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3,
    random_seed=42,
    verbose=False
)
evaluate_and_store_model('CatBoost', cat_model, X_train, y_train, X_test, y_test)

In [ ]:
# 3. FLAML AutoML
print("\n🚀 ENTRENANDO FLAML AUTOML")
print("=" * 50)

try:
    flaml_model = AutoML()
    
    # Configuración para series temporales financieras
    flaml_settings = {
        'time_budget': 300,  # 5 minutos
        'metric': 'r2',  # Usaremos R2 como proxy
        'task': 'regression',
        'estimator_list': ['lgbm', 'xgboost', 'catboost', 'rf', 'extra_tree'],
        'eval_method': 'cv',
        'split_type': 'time',  # Importante para series temporales
        'n_splits': 3,
        'verbose': 1,
        'seed': 42
    }
    
    flaml_model.fit(X_train, y_train, **flaml_settings)
    
    evaluate_and_store_model('FLAML_AutoML', flaml_model, X_train, y_train, X_test, y_test)
    
    print(f"🏆 Mejor modelo FLAML: {flaml_model.best_estimator}")
    print(f"🏆 Mejores hiperparámetros: {flaml_model.best_config}")
    
except Exception as e:
    print(f"❌ Error con FLAML: {e}")

In [ ]:
# 4. AutoGluon (si está disponible)
print("\n🚀 ENTRENANDO AUTOGLUON")
print("=" * 50)

try:
    # Preparar datos para AutoGluon
    train_data = X_train.copy()
    train_data[target_col] = y_train
    
    test_data = X_test.copy()
    test_data[target_col] = y_test
    
    # Configurar AutoGluon
    predictor = TabularPredictor(
        label=target_col,
        problem_type='regression',
        eval_metric='r2',
        path='./autogluon_models'
    )
    
    # Entrenar con tiempo limitado
    predictor.fit(
        train_data,
        time_limit=300,  # 5 minutos
        presets='medium_quality',
        verbosity=2
    )
    
    # Crear wrapper para compatibilidad
    class AutoGluonWrapper:
        def __init__(self, predictor):
            self.predictor = predictor
        
        def predict(self, X):
            return self.predictor.predict(X)
        
        def fit(self, X, y):
            pass  # Ya está entrenado
    
    ag_wrapper = AutoGluonWrapper(predictor)
    
    # Evaluar
    y_pred_train_ag = predictor.predict(X_train)
    y_pred_test_ag = predictor.predict(X_test)
    
    train_score_ag = competition_score(y_train, y_pred_train_ag)
    test_score_ag = competition_score(y_test, y_pred_test_ag)
    
    models_results['AutoGluon'] = {
        'model': ag_wrapper,
        'train_score': train_score_ag,
        'test_score': test_score_ag,
        'predictions_train': y_pred_train_ag,
        'predictions_test': y_pred_test_ag
    }
    
    print(f"   ✅ AutoGluon - Train Score: {train_score_ag:.6f}, Test Score: {test_score_ag:.6f}")
    
    # Mostrar leaderboard
    leaderboard = predictor.leaderboard(test_data, silent=True)
    print("\n🏆 AutoGluon Leaderboard:")
    print(leaderboard.head())
    
except Exception as e:
    print(f"❌ Error con AutoGluon: {e}")

## 6. Ensemble y Optimización

In [ ]:
# Resumen de resultados
print("📊 RESUMEN DE RESULTADOS DE MODELOS")
print("=" * 60)

results_df = pd.DataFrame({
    'Model': list(models_results.keys()),
    'Train_Score': [models_results[model]['train_score'] for model in models_results.keys()],
    'Test_Score': [models_results[model]['test_score'] for model in models_results.keys()],
    'Train_R2': [models_results[model].get('train_r2', 0) for model in models_results.keys()],
    'Test_R2': [models_results[model].get('test_r2', 0) for model in models_results.keys()]
})

results_df = results_df.sort_values('Test_Score', ascending=False)
print(results_df.to_string(index=False))

# Visualización de resultados
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Scores de competencia
x_pos = np.arange(len(results_df))
axes[0].bar(x_pos - 0.2, results_df['Train_Score'], 0.4, label='Train', alpha=0.8)
axes[0].bar(x_pos + 0.2, results_df['Test_Score'], 0.4, label='Test', alpha=0.8)
axes[0].set_xlabel('Modelos')
axes[0].set_ylabel('Competition Score')
axes[0].set_title('Scores de Competencia por Modelo')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(results_df['Model'], rotation=45)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# R2 scores
axes[1].bar(x_pos - 0.2, results_df['Train_R2'], 0.4, label='Train R2', alpha=0.8)
axes[1].bar(x_pos + 0.2, results_df['Test_R2'], 0.4, label='Test R2', alpha=0.8)
axes[1].set_xlabel('Modelos')
axes[1].set_ylabel('R² Score')
axes[1].set_title('R² Scores por Modelo')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(results_df['Model'], rotation=45)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Crear ensemble de los mejores modelos
def create_ensemble(models_dict, top_n=3, method='weighted_average'):
    """
    Crea un ensemble de los mejores modelos
    """
    # Seleccionar top N modelos por test score
    sorted_models = sorted(models_dict.items(), 
                          key=lambda x: x[1]['test_score'], 
                          reverse=True)
    
    top_models = sorted_models[:top_n]
    print(f"🏆 Top {top_n} modelos para ensemble:")
    for name, results in top_models:
        print(f"   {name}: {results['test_score']:.6f}")
    
    if method == 'simple_average':
        # Promedio simple
        weights = [1/len(top_models)] * len(top_models)
    elif method == 'weighted_average':
        # Pesos basados en performance
        scores = [results['test_score'] for _, results in top_models]
        # Normalizar scores para que sean positivos
        min_score = min(scores)
        adjusted_scores = [score - min_score + 0.001 for score in scores]
        total_score = sum(adjusted_scores)
        weights = [score / total_score for score in adjusted_scores]
    
    print(f"📊 Pesos del ensemble: {[f'{w:.3f}' for w in weights]}")
    
    class EnsembleModel:
        def __init__(self, models, weights):
            self.models = [(name, results['model']) for name, results in models]
            self.weights = weights
        
        def predict(self, X):
            predictions = []
            for (name, model), weight in zip(self.models, self.weights):
                try:
                    pred = model.predict(X)
                    predictions.append(pred * weight)
                except Exception as e:
                    print(f"Error con modelo {name}: {e}")
                    predictions.append(np.zeros(len(X)) * weight)
            
            return np.sum(predictions, axis=0)
        
        def fit(self, X, y):
            pass  # Los modelos ya están entrenados
    
    ensemble = EnsembleModel(top_models, weights)
    return ensemble

# Crear ensemble
ensemble_model = create_ensemble(models_results, top_n=3, method='weighted_average')

# Evaluar ensemble
evaluate_and_store_model('Ensemble_Top3', ensemble_model, X_train, y_train, X_test, y_test)

print("\n✅ Ensemble creado y evaluado")

## 7. Validación Temporal y Testing

In [ ]:
# Validación cruzada temporal para el mejor modelo
best_model_name = max(models_results.keys(), key=lambda x: models_results[x]['test_score'])
best_model = models_results[best_model_name]['model']

print(f"🏆 Mejor modelo: {best_model_name}")
print(f"🏆 Score: {models_results[best_model_name]['test_score']:.6f}")

# Validación cruzada temporal
print("\n🔄 Realizando validación cruzada temporal...")
cv_scores = evaluate_model_temporal(best_model, X_final, y_final, n_splits=5)

print(f"📊 CV Scores: {cv_scores}")
print(f"📊 CV Mean: {cv_scores.mean():.6f} ± {cv_scores.std():.6f}")

# Visualización de CV scores
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(cv_scores) + 1), cv_scores, 'bo-', linewidth=2, markersize=8)
plt.axhline(y=cv_scores.mean(), color='r', linestyle='--', alpha=0.7, label=f'Media: {cv_scores.mean():.6f}')
plt.fill_between(range(1, len(cv_scores) + 1), 
                 cv_scores.mean() - cv_scores.std(), 
                 cv_scores.mean() + cv_scores.std(), 
                 alpha=0.2, color='red')
plt.xlabel('Fold')
plt.ylabel('Competition Score')
plt.title(f'Validación Cruzada Temporal - {best_model_name}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Análisis de predicciones
def analyze_predictions(y_true, y_pred, title="Análisis de Predicciones"):
    """
    Analiza la calidad de las predicciones
    """
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle(title, fontsize=16, fontweight='bold')
    
    # Scatter plot: Predicciones vs Reales
    axes[0, 0].scatter(y_true, y_pred, alpha=0.6, s=20)
    axes[0, 0].plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', lw=2)
    axes[0, 0].set_xlabel('Valores Reales')
    axes[0, 0].set_ylabel('Predicciones')
    axes[0, 0].set_title('Predicciones vs Reales')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Residuos
    residuals = y_true - y_pred
    axes[0, 1].scatter(y_pred, residuals, alpha=0.6, s=20)
    axes[0, 1].axhline(y=0, color='r', linestyle='--')
    axes[0, 1].set_xlabel('Predicciones')
    axes[0, 1].set_ylabel('Residuos')
    axes[0, 1].set_title('Gráfico de Residuos')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Distribución de residuos
    axes[1, 0].hist(residuals, bins=50, alpha=0.7, edgecolor='black')
    axes[1, 0].axvline(x=0, color='r', linestyle='--')
    axes[1, 0].set_xlabel('Residuos')
    axes[1, 0].set_ylabel('Frecuencia')
    axes[1, 0].set_title('Distribución de Residuos')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Serie temporal de predicciones
    indices = range(len(y_true))
    axes[1, 1].plot(indices, y_true, label='Real', alpha=0.7, linewidth=1)
    axes[1, 1].plot(indices, y_pred, label='Predicción', alpha=0.7, linewidth=1)
    axes[1, 1].set_xlabel('Tiempo')
    axes[1, 1].set_ylabel('Valor')
    axes[1, 1].set_title('Serie Temporal: Real vs Predicción')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Estadísticas
    print(f"📊 ESTADÍSTICAS DE PREDICCIÓN")
    print("=" * 40)
    print(f"MSE: {mean_squared_error(y_true, y_pred):.6f}")
    print(f"MAE: {mean_absolute_error(y_true, y_pred):.6f}")
    print(f"R²: {r2_score(y_true, y_pred):.6f}")
    print(f"Competition Score: {competition_score(y_true, y_pred):.6f}")
    print(f"Correlación: {np.corrcoef(y_true, y_pred)[0, 1]:.6f}")

# Analizar predicciones del mejor modelo
best_predictions = models_results[best_model_name]['predictions_test']
analyze_predictions(y_test, best_predictions, f"Análisis de Predicciones - {best_model_name}")

## 8. Preparación para Submission

In [ ]:
# Función de predicción final para submission
def create_prediction_function(model, scaler, selected_features, feature_cols, target_col):
    """
    Crea una función de predicción que incluye todo el pipeline
    """
    def predict_function(test_df):
        """
        Función de predicción para el API de Kaggle
        """
        # Aplicar ingeniería de características
        test_enhanced = create_technical_features(test_df, feature_cols, target_col)
        
        # Seleccionar características
        test_features = test_enhanced[selected_features]
        
        # Manejar valores faltantes
        test_features = test_features.fillna(test_features.mean())
        
        # Escalar si es necesario
        if scaler is not None:
            test_features_scaled = pd.DataFrame(
                scaler.transform(test_features),
                columns=test_features.columns,
                index=test_features.index
            )
            predictions = model.predict(test_features_scaled)
        else:
            predictions = model.predict(test_features)
        
        # Aplicar límites de posición
        predictions = np.clip(predictions, -6.0, 6.0)
        
        return predictions
    
    return predict_function

# Crear función de predicción final
final_predict = create_prediction_function(
    model=best_model,
    scaler=scaler if best_model_name in ['Ridge', 'RandomForest', 'GradientBoosting'] else None,
    selected_features=all_selected,
    feature_cols=feature_cols,
    target_col=target_col
)

print(f"✅ Función de predicción final creada usando {best_model_name}")

In [ ]:
# Código de submission para Kaggle
submission_code = f'''
# Hull Tactical Market Prediction - AutoML Solution
# Mejor modelo: {best_model_name}
# Score de validación: {models_results[best_model_name]['test_score']:.6f}

import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import RobustScaler
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge

# Cargar modelo y componentes (estos se guardarían en archivos)
# model = pickle.load(open('best_model.pkl', 'rb'))
# scaler = pickle.load(open('scaler.pkl', 'rb'))
# selected_features = pickle.load(open('selected_features.pkl', 'rb'))

def create_technical_features(df, feature_cols, target_col):
    """Función de ingeniería de características"""
    # [Código de ingeniería de características aquí]
    pass

def predict(test_df):
    """Función de predicción principal"""
    try:
        # Aplicar ingeniería de características
        test_enhanced = create_technical_features(test_df, feature_cols, target_col)
        
        # Seleccionar y preparar características
        test_features = test_enhanced[selected_features]
        test_features = test_features.fillna(test_features.mean())
        
        # Hacer predicciones
        predictions = model.predict(test_features)
        
        # Aplicar límites
        predictions = np.clip(predictions, -6.0, 6.0)
        
        return predictions
        
    except Exception as e:
        print(f"Error en predicción: {{e}}")
        # Fallback: retornar predicciones conservadoras
        return np.zeros(len(test_df))

# Ejecutar evaluación
import kaggle_evaluation.hull_tactical_market_prediction as evaluation
evaluation.run(predict)
'''

print("📝 CÓDIGO DE SUBMISSION GENERADO")
print("=" * 50)
print("El código anterior debe ser adaptado para incluir:")
print("1. Guardar el modelo entrenado")
print("2. Guardar el scaler y características seleccionadas")
print("3. Implementar la función completa de ingeniería de características")
print("4. Manejar errores y casos edge")

In [ ]:
# Guardar componentes del modelo
import pickle
import joblib

# Crear directorio para modelos
import os
os.makedirs('model_artifacts', exist_ok=True)

# Guardar modelo
model_path = f'model_artifacts/best_model_{best_model_name.lower()}.pkl'
joblib.dump(best_model, model_path)
print(f"✅ Modelo guardado en: {model_path}")

# Guardar scaler
if scaler is not None:
    scaler_path = 'model_artifacts/scaler.pkl'
    joblib.dump(scaler, scaler_path)
    print(f"✅ Scaler guardado en: {scaler_path}")

# Guardar características seleccionadas
features_path = 'model_artifacts/selected_features.pkl'
with open(features_path, 'wb') as f:
    pickle.dump(all_selected, f)
print(f"✅ Características guardadas en: {features_path}")

# Guardar características originales
original_features_path = 'model_artifacts/original_features.pkl'
with open(original_features_path, 'wb') as f:
    pickle.dump(feature_cols, f)
print(f"✅ Características originales guardadas en: {original_features_path}")

# Guardar resumen de resultados
results_path = 'model_artifacts/model_results.csv'
results_df.to_csv(results_path, index=False)
print(f"✅ Resultados guardados en: {results_path}")

print("\n🎯 RESUMEN FINAL")
print("=" * 50)
print(f"🏆 Mejor modelo: {best_model_name}")
print(f"🏆 Score de validación: {models_results[best_model_name]['test_score']:.6f}")
print(f"📊 Características utilizadas: {len(all_selected)}")
print(f"📊 Datos de entrenamiento: {X_train.shape[0]:,} muestras")
print(f"📊 Datos de prueba: {X_test.shape[0]:,} muestras")

print("\n✅ Notebook completado exitosamente!")
print("\n📋 PRÓXIMOS PASOS:")
print("1. Revisar y optimizar la función de ingeniería de características")
print("2. Implementar el código de submission completo")
print("3. Probar con datos de validación adicionales")
print("4. Considerar ensemble con más modelos")
print("5. Optimizar hiperparámetros con más tiempo")